# SignSync — SVM Training (Google Colab)

**No GPU needed.** Free Colab CPU runtime finishes in ~5–10 minutes.

Run each cell in order ↓

## Step 1 — Install dependencies

In [ ]:
!pip install -q mediapipe scikit-learn opencv-python-headless tqdm joblib kagglehub matplotlib

## Step 2 — Upload training script

Upload `train_svm_standalone.py` from your project (`src/app/core/ml/sign_model_pytorch/`)

In [ ]:
from google.colab import files

files.upload()  # upload train_svm_standalone.py
!ls -lh train_svm_standalone.py

## Step 3 — Config (edit before running)

In [ ]:
# ── Edit these if needed ────────────────────────────────────────────────────
MAX_PER_CLASS = 1000  # ↑ from 500 → more data improves B/W separation
SYNTHETIC_PER_SIGN = 600  # synthetic samples per emergency sign
TEST_MODE = False  # True = quick smoke-test (3 classes only)
# ────────────────────────────────────────────────────────────────────────────

import sys

args = [
    "train_svm_standalone.py",
    "--max-per-class",
    str(MAX_PER_CLASS),
    "--synthetic-per-sign",
    str(SYNTHETIC_PER_SIGN),
]
if TEST_MODE:
    args.append("--test")
sys.argv = args
print("Config:", args)
print("Features: 82-D (63 coords + 19 geometric — tip distances, extensions, spreads)")
print("Pipeline: StandardScaler → SVM(RBF, C=50, class_weight=balanced)  [no PCA]")

## What changed from v1 → v2

| # | Old | New | Why |
|---|-----|-----|-----|
| 1 | 63-D coords only | **82-D** (63 coords + 19 geometric) | Tip distances & spreads let SVM directly see B vs W difference |
| 2 | `PCA(95%)` → 9-D | **No PCA** | PCA was discarding the exact dims that separate B from W |
| 3 | `C=10` | **`C=50`** | Tighter boundary for similar-looking signs |
| 4 | No class weighting | **`class_weight='balanced'`** | Prevents majority classes from dominating |
| 5 | `max_per_class=500` | **`max_per_class=1000`** | More real data for hard sign pairs |
| 6 | XY rotation only | **XY + Z-axis rotation** | More realistic hand depth augmentation |

### The 19 new geometric features
```
10  pairwise tip distances  → B: tips clustered;  W: tips spread far apart
 5  finger extension scores → how far each finger extends above its MCP
 4  adjacent tip x-spreads  → direct left/right spread between fingers
```
> **Note:** After retraining, place the new `sign_language_svm.joblib` in  
> `src/app/core/ml/sign_model_pytorch/trained_model/` and rebuild Docker: `docker compose up --build`

## Step 4 — Train!

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("svm_train", "train_svm_standalone.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
module.main()

## Step 5 — Download model + report artifacts

In [ ]:
import os

from google.colab import files

base = "trained_model"
files_to_download = [
    f"{base}/sign_language_svm.joblib",
    f"{base}/class_names_svm.json",
    f"{base}/confusion_matrix.png",
    f"{base}/confusion_matrix_normalized.png",
    f"{base}/confusion_matrix.csv",
    f"{base}/learning_curve.png",
    f"{base}/learning_curve.csv",
    f"{base}/per_class_f1_scores.png",
    f"{base}/per_class_precision_recall_f1.png",
    f"{base}/class_support_distribution.png",
    f"{base}/prediction_confidence_histogram.png",
    f"{base}/per_class_metrics.csv",
    f"{base}/metrics_summary.json",
]

for f in files_to_download:
    if os.path.exists(f):
        files.download(f)
        print(f"⬇️  Downloaded {f}")
    else:
        print(f"⚠️  Not found: {f}")

print("\nCopy model files to:")
print("  backend/src/app/core/ml/sign_model_pytorch/trained_model/")